# Part 3 — Churn Prediction Model & Model Card

This notebook contains the data loading, pre-processing, training of Baseline (Logistic Regression) and XGBoost models, threshold optimization using business cost rules, evaluation, feature importance, and error analysis.

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
DATA_DIR = r"../data"
print("Libraries loaded successfully.")

### 1. Load Data and Train-Val-Test Splits

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "rfm_modeling_snapshot.csv"))

# Pre-processing: Fill missing categoricals
df["loyalty_tier"] = df["loyalty_tier"].fillna("Not Enrolled")

# Separate split
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

print(f"Train set size: {train_df.shape[0]}")
print(f"Validation set size: {val_df.shape[0]}")
print(f"Test set size: {test_df.shape[0]}")

In [ ]:
# Features definition
cat_cols = ["city_tier", "age_group", "acquisition_channel", "loyalty_tier", "preferred_category", "marketing_consent"]
num_cols = [
    "recency_days", "frequency_180d", "monetary_180d", "return_rate_180d", 
    "avg_discount_pct_180d", "avg_rating_180d", "category_diversity_180d", 
    "ticket_count_90d", "negative_ticket_rate_90d", "avg_resolution_hours_90d", 
    "days_since_signup", "sessions_30d", "product_views_30d", "cart_adds_30d", 
    "wishlist_adds_30d", "abandoned_carts_30d", "email_opens_30d", "campaign_clicks_30d", 
    "last_visit_days_ago"
]
target_col = "churn_next_60d"

X_train, y_train = train_df[cat_cols + num_cols], train_df[target_col]
X_val, y_val = val_df[cat_cols + num_cols], val_df[target_col]
X_test, y_test = test_df[cat_cols + num_cols], test_df[target_col]

### 2. Preprocessing & Models Setup

In [ ]:
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])

### 3. Model Training & Validation

In [ ]:
# Logistic Regression Baseline
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipeline.fit(X_train, y_train)
y_val_pred_lr = lr_pipeline.predict(X_val)
y_val_prob_lr = lr_pipeline.predict_proba(X_val)[:, 1]

# Random Forest
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42))
])
rf_pipeline.fit(X_train, y_train)
y_val_pred_rf = rf_pipeline.predict(X_val)
y_val_prob_rf = rf_pipeline.predict_proba(X_val)[:, 1]

# XGBoost
X_train_trans = preprocessor.fit_transform(X_train)
X_val_trans = preprocessor.transform(X_val)
xgb = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42, eval_metric="logloss")
xgb.fit(X_train_trans, y_train)
y_val_pred_xgb = xgb.predict(X_val_trans)
y_val_prob_xgb = xgb.predict_proba(X_val_trans)[:, 1]

In [ ]:
def eval_model(name, y_true, y_pred, y_prob):
    print(f"\n=== {name} Validation Metrics ===")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_true, y_prob):.4f}")

eval_model("Logistic Regression", y_val, y_val_pred_lr, y_val_prob_lr)
eval_model("Random Forest", y_val, y_val_pred_rf, y_val_prob_rf)
eval_model("XGBoost", y_val, y_val_pred_xgb, y_val_prob_xgb)

### 4. Threshold Optimization via Business Logic

In [ ]:
thresholds = np.linspace(0.1, 0.9, 81)
min_cost = float('inf')
best_threshold = 0.5

for t in thresholds:
    preds = (y_val_prob_xgb >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    cost = fp * 150 + fn * 1250
    if cost < min_cost:
        min_cost = cost
        best_threshold = t

print(f"Optimal Business Threshold: {best_threshold:.2f}")
print(f"Expected Validation Cost at 0.50: {confusion_matrix(y_val, y_val_pred_xgb).ravel()[1]*150 + confusion_matrix(y_val, y_val_pred_xgb).ravel()[2]*1250} INR")
print(f"Expected Validation Cost at {best_threshold:.2f}: {min_cost} INR")

### 5. Final Evaluation on Test Set

In [ ]:
X_test_trans = preprocessor.transform(X_test)
y_test_pred_xgb = xgb.predict(X_test_trans)
y_test_prob_xgb = xgb.predict_proba(X_test_trans)[:, 1]

eval_model("XGBoost (Test Set)", y_test, y_test_pred_xgb, y_test_prob_xgb)

### 6. Feature Importance

In [ ]:
cat_encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]
cat_features = list(cat_encoder.get_feature_names_out(cat_cols))
all_features = num_cols + cat_features

feat_imp = pd.DataFrame({
    "feature": all_features,
    "importance": xgb.feature_importances_
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp.head(15), x="importance", y="feature", palette="viridis")
plt.title("Top 15 Feature Importances (XGBoost)")
plt.show()